In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q "timm==0.9.16" "transformers" "albumentations==1.4.18"

import os, gc, math, random, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel
import timm
from sklearn.metrics import f1_score, classification_report
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")
SEED, N_CLS = 42, 8
device = torch.device("cuda")

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(SEED)
CLASSES = ['Drought','Earthquake','Flood','Human Damage',
           'Landslides','Non Disaster','Tropical Storm','Wildfire']
LABEL2ID = {l:i for i,l in enumerate(CLASSES)}
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)
EPS = 1e-12
print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.0/224.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 78.9 MB/s eta 0:00:00:00:01:01m
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numb

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


GPU: Tesla T4


In [2]:
COMP     = Path("/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0")
S3D_ROOT = Path("/kaggle/input/datasets/rafiurrahman01/oof-s3d-complete")
OOF_DIR  = S3D_ROOT / "oof"

WORK = Path("/kaggle/working")
OUT  = WORK / "joint_mm"; OUT.mkdir(parents=True, exist_ok=True)
CKPT = WORK / "ckpts"; CKPT.mkdir(exist_ok=True)

train = pd.read_csv(next(COMP.rglob("Disaster_train.csv")), encoding="utf-8-sig")
test  = pd.read_csv(next(COMP.rglob("Disaster_test.csv")),  encoding="utf-8-sig")
train.columns = [c.strip() for c in train.columns]
test.columns  = [c.strip() for c in test.columns]
train["label"] = train["category"].map(LABEL2ID)

folds_df = pd.read_csv(S3D_ROOT / "meta" / "folds_canonical.csv")
train = train.merge(folds_df[["image_id","fold"]], on="image_id", how="left")
train["fold"] = train["fold"].astype(int)

TRAIN_IMG = next(COMP.rglob("Train"))
TEST_IMG  = next(COMP.rglob("Test"))

def resolve_img(img_id, is_test):
    base = TEST_IMG if is_test else TRAIN_IMG
    for ext in [".jpg",".jpeg",".png",".JPG",".PNG"]:
        p = base / f"{img_id}{ext}"
        if p.exists(): return p
    for sub in base.iterdir():
        if sub.is_dir():
            for ext in [".jpg",".jpeg",".png"]:
                p = sub / f"{img_id}{ext}"
                if p.exists(): return p
    raise FileNotFoundError(img_id)

train["img_path"] = train["image_id"].apply(lambda x: resolve_img(x, False))
test["img_path"]  = test["image_id"].apply(lambda x: resolve_img(x, True))
print(f"Train: {train.shape} | Test: {test.shape}")

def pad_resize(img_pil, size):
    img = np.array(img_pil.convert("RGB"))
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = int(round(h*s)), int(round(w*s))
    img = A.Resize(nh, nw)(image=img)["image"]
    pad_h, pad_w = size-nh, size-nw
    return np.pad(img, ((pad_h//2,pad_h-pad_h//2),(pad_w//2,pad_w-pad_w//2),(0,0)), constant_values=0)

Train: (6323, 6) | Test: (1580, 3)


In [3]:
JM_CFG = {
    "img_model":    "convnextv2_base.fcmae_ft_in22k_in1k_384",   # 89M, faster than EVA
    "text_model":   "google/muril-large-cased",
    "img_size":     384,
    "max_len":      256,
    "img_dim":      1024,    # convnextv2_base output
    "text_dim":     1024,    # MuRIL-Large
    "fusion_dim":   512,
    "n_heads":      8,
    "batch_size":   4,
    "grad_accum":   8,
    "epochs":       5,
    "lr_image":     1e-5,
    "lr_text":      8e-6,
    "lr_head":      5e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "label_smooth": 0.05,
    "drop_path":    0.10,
    "tag":          "joint_cnxv2_muril",
    "run_id":       f"jmm_{datetime.now():%Y%m%d_%H%M}",
}

tokenizer = AutoTokenizer.from_pretrained(JM_CFG["text_model"])

train_tfm = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.ShiftScaleRotate(0.05,0.10,12,p=1.0),
        A.Affine(scale=(0.92,1.08),translate_percent=0.04,rotate=(-10,10),p=1.0),
    ],p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(0.15,0.15,p=1.0),
        A.HueSaturationValue(10,15,10,p=1.0),
    ],p=0.4),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    A.Normalize(MEAN,STD), ToTensorV2(),
])
val_tfm = A.Compose([A.Normalize(MEAN,STD), ToTensorV2()])

class MMDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, img_size, tfm, is_test=False):
        self.df=df.reset_index(drop=True); self.tok=tokenizer
        self.max_len=max_len; self.img_size=img_size; self.tfm=tfm; self.is_test=is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = pad_resize(Image.open(row["img_path"]), self.img_size)
        img = self.tfm(image=img)["image"]
        enc = self.tok(str(row["context"]), max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        item = {k:v.squeeze(0) for k,v in enc.items()}
        item["image"] = img
        if not self.is_test:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

config.json:   0%|          | 0.00/406 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
class JointMM(nn.Module):
    def __init__(self, cfg, n_cls=N_CLS):
        super().__init__()
        # Image encoder — fine-tuned, with gradient checkpointing
        self.img_enc = timm.create_model(
            cfg["img_model"], pretrained=True, num_classes=0,
            drop_path_rate=cfg["drop_path"],
        )
        if hasattr(self.img_enc, "set_grad_checkpointing"):
            try:
                self.img_enc.set_grad_checkpointing(enable=True)
            except Exception:
                pass

        # Text encoder — fine-tuned
        self.text_enc = AutoModel.from_pretrained(cfg["text_model"])
        if hasattr(self.text_enc, "gradient_checkpointing_enable"):
            self.text_enc.gradient_checkpointing_enable()

        fd = cfg["fusion_dim"]
        self.img_proj  = nn.Sequential(nn.LayerNorm(cfg["img_dim"]),
                                       nn.Linear(cfg["img_dim"], fd), nn.GELU())
        self.text_proj = nn.Sequential(nn.LayerNorm(cfg["text_dim"]),
                                       nn.Linear(cfg["text_dim"], fd), nn.GELU())
        self.cross_attn = nn.MultiheadAttention(fd, cfg["n_heads"],
                                                 dropout=0.1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.LayerNorm(fd*2),
            nn.Dropout(0.2),
            nn.Linear(fd*2, n_cls),
        )

    def forward(self, image, input_ids, attention_mask, token_type_ids=None):
        # Image features
        img_f = self.img_enc(image)  # (B, img_dim)

        # Text features (mean pool)
        kw = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None: kw["token_type_ids"] = token_type_ids
        t_out = self.text_enc(**kw).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        txt_f = (t_out * m).sum(1) / m.sum(1).clamp(min=1e-9)

        img_q = self.img_proj(img_f).unsqueeze(1)
        txt_q = self.text_proj(txt_f).unsqueeze(1)
        attn_out, _ = self.cross_attn(query=txt_q, key=img_q, value=img_q)
        fused = torch.cat([attn_out.squeeze(1), self.text_proj(txt_f)], dim=-1)
        return self.classifier(fused)

def cosine_warmup(optim, n_warmup, n_total):
    def fn(step):
        if step < n_warmup: return step / max(1, n_warmup)
        prog = (step - n_warmup) / max(1, n_total - n_warmup)
        return max(0.0, 0.5*(1 + math.cos(math.pi*prog)))
    return torch.optim.lr_scheduler.LambdaLR(optim, fn)

def soft_ce(logits, soft_targets, smoothing=0.05):
    n = logits.size(1)
    lp = F.log_softmax(logits.float(), dim=-1)
    soft = soft_targets * (1-smoothing) + smoothing/n
    return -(soft * lp).sum(-1).mean()

print("Model ready ✓")

Model ready ✓


In [ ]:
def infer_test_jmm(cfg, ckpts):
    loader = DataLoader(
        MMDataset(test, tokenizer, cfg["max_len"], cfg["img_size"], val_tfm, is_test=True),
        batch_size=cfg["batch_size"]*2, shuffle=False, num_workers=2, pin_memory=True)
    out = np.zeros((len(test), N_CLS), np.float32)
    for ckpt in ckpts:
        model = JointMM(cfg).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device))
        model.eval()
        fp = []
        with torch.no_grad(), autocast(dtype=torch.float16):
            for batch in tqdm(loader, desc=f"infer {Path(ckpt).name}", leave=False):
                img=batch["image"].to(device); ids=batch["input_ids"].to(device)
                mask=batch["attention_mask"].to(device)
                ttid = batch.get("token_type_ids")
                if ttid is not None: ttid = ttid.to(device)
                p = F.softmax(model(img,ids,mask,ttid).float(), -1)
                fp.append(p.cpu().numpy())
        out += np.concatenate(fp) / len(ckpts)
        del model; gc.collect(); torch.cuda.empty_cache()
    return out

ckpts = sorted(CKPT.glob(f"{JM_CFG['tag']}_f*.pth"))
test_jmm = infer_test_jmm(JM_CFG, ckpts)
np.save(OUT/f"{JM_CFG['run_id']}_oof.npy", oof_jmm)
np.save(OUT/f"{JM_CFG['run_id']}_test_probs.npy", test_jmm)

# Blend with S3C best
stack_s3e = np.load(OOF_DIR/"stack_oof_s3e.npy")
final_oof = np.load(OOF_DIR/"final_oof_s2.npy")
new_bias  = np.load(OOF_DIR/"new_bias_s3e.npy")
stack_test= np.load(OOF_DIR/"stack_test_s3e.npy")
final_test= np.load(OOF_DIR/"final_test_s2.npy")
spec_oof  = np.load(OOF_DIR/"spec_eqflls_20260512_1557_oof.npy")
spec_test = np.load(OOF_DIR/"spec_eqflls_20260512_1557_test_probs.npy")
nan_rows  = np.where(np.isnan(final_test).any(1))[0]
final_test[nan_rows] = test_jmm[nan_rows]

SPEC_IDS=[1,2,4]; spec_mask=np.isin(train["label"].values,SPEC_IDS)
spec_full = np.zeros((len(train),3), np.float32); spec_full[spec_mask]=spec_oof
def apply_spec(p8, p3, beta):
    out = p8.copy()
    for li,gi in enumerate(SPEC_IDS): out[:,gi]*=(p3[:,li]**beta)
    return out/out.sum(1,keepdims=True).clip(min=1e-12)

base_oof  = 0.3*stack_s3e + 0.7*final_oof
base_test = 0.3*stack_test+ 0.7*final_test
y = train["label"].values

best_f, best = -1, None
for a in np.linspace(0,0.5,11):
    for b in np.linspace(0.3,1.5,7):
        blend = a*oof_jmm + (1-a)*base_oof
        corr = apply_spec(blend, spec_full, b)
        f = f1_score(y, (np.log(corr+EPS)+new_bias).argmax(1), average="macro")
        if f>best_f: best_f, best = f, (a,b)
a,b = best
print(f"Best JMM weight={a:.2f}, beta={b:.2f}, OOF F1={best_f:.5f}")

tb = a*test_jmm + (1-a)*base_test
tc = apply_spec(tb, spec_test, b)
tp = (np.log(tc+EPS)+new_bias).argmax(1)
for i, ctx in enumerate(test["context"].astype(str)):
    if any(e in ctx for e in ['📍','😍','🥰','📌','✨']): tp[i]=LABEL2ID['Non Disaster']
    elif any(e in ctx for e in ['🚨','⚠']): tp[i]=LABEL2ID['Landslides']
    elif any(e in ctx for e in ['♻','🚱']): tp[i]=LABEL2ID['Human Damage']

sub = pd.DataFrame({"image_id":test["image_id"], "category":[CLASSES[p] for p in tp]})
sub.to_csv(WORK/"submission_jointmm.csv", index=False)
print(f"Saved | Submit if OOF > 0.99289 (current best blend)")

In [1]:
import os, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize

# === CHANGE ONLY IF YOUR DATASET NAME IS DIFFERENT ===
OOF_INPUT = Path('/kaggle/input/datasets/rafiurrahman01/oof-s3d-complete')          # uploaded zip contents
COMP_ROOT = Path('/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0')

WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
LABELS = ['Drought','Earthquake','Flood','Human Damage','Landslides',
          'Non Disaster','Tropical Storm','Wildfire']
LABEL2ID = {l:i for i,l in enumerate(LABELS)}
EPS = 1e-12

# Verify paths
print("OOF folder contents:")
for f in sorted((OOF_INPUT/'oof').glob('*.npy'))[:5]:
    print(" ", f.name)
print("\nMeta folder:")
for f in sorted((OOF_INPUT/'meta').glob('*.csv')):
    print(" ", f.name)
print("\nCompetition data:")
for f in sorted(COMP_ROOT.glob('*.csv')):
    print(" ", f.name)

OOF folder contents:
  blend_weights_s2.npy
  class_bias_s2.npy
  final_oof_s2.npy
  final_test_s2.npy
  img_eva02L_448_20260510_1049_oof_partial.npy

Meta folder:
  folds_canonical.csv

Competition data:


In [4]:
folds = pd.read_csv(OOF_INPUT/'meta/folds_canonical.csv')
y_true = folds['label'].values
test = pd.read_csv('/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0/Dataset/CSV Dataset/Disaster_test.csv')
print(f"Train rows: {len(folds)}  Test rows: {len(test)}")

# Find the submission column name (handles the 'categry' typo)
sample_sub = pd.read_csv(COMP_ROOT/'sample_submission.csv') if (COMP_ROOT/'sample_submission.csv').exists() \
             else pd.read_csv('/kaggle/input/competitions/detect-the-disaster-intra-cuet-ml-contest-2-0/Dataset/sample submission.csv')
SUB_LABEL_COL = [c for c in sample_sub.columns if c != 'image_id'][0]
print(f"Submission label column: '{SUB_LABEL_COL}'")

OOF_DIR = OOF_INPUT/'oof'
base_oof = {
    'bb_base':       np.load(OOF_DIR/'text_bbB_s123_20260509_0550_oof.npy'),
    'bb_multi':      np.load(OOF_DIR/'text_bbmulti_20260509_0837_oof.npy'),
    'muril_L':       np.load(OOF_DIR/'text_murilL_20260509_0210_oof.npy'),
    'eva02_L':       np.load(OOF_DIR/'img_eva02L_448_20260511_0327_oof.npy'),
    'mm_eva_muril':  np.load(OOF_DIR/'mm_eva_muril_20260512_1041_oof.npy'),
    'pl_muril':      np.load(OOF_DIR/'pl_muril_20260512_1725_oof.npy'),
}
base_te = {
    'bb_base':       np.load(OOF_DIR/'text_bbB_s123_20260509_0550_test_probs.npy'),
    'bb_multi':      np.load(OOF_DIR/'text_bbmulti_20260509_0837_test_probs.npy'),
    'muril_L':       np.load(OOF_DIR/'text_murilL_20260509_0210_test_probs.npy'),
    'eva02_L':       np.load(OOF_DIR/'img_eva02L_448_20260511_0327_test_probs.npy'),
    'mm_eva_muril':  np.load(OOF_DIR/'mm_eva_muril_20260512_1041_test_probs.npy'),
    'pl_muril':      np.load(OOF_DIR/'pl_muril_20260512_1725_test_probs.npy'),
}
stack_oof = np.load(OOF_DIR/'stack_oof_s3e.npy')
stack_te  = np.load(OOF_DIR/'stack_test_s3e.npy')

# Individual OOF F1 scores
print("\nIndividual OOF F1:")
for n, o in base_oof.items():
    print(f"  {n:18s} = {f1_score(y_true, o.argmax(1), average='macro'):.5f}")
print(f"  stack_s3e          = {f1_score(y_true, stack_oof.argmax(1), average='macro'):.5f}")

Train rows: 6323  Test rows: 1580
Submission label column: 'category'

Individual OOF F1:
  bb_base            = 0.96587
  bb_multi           = 0.96656
  muril_L            = 0.97556
  eva02_L            = 0.96714
  mm_eva_muril       = 0.99212
  pl_muril           = 0.97525
  stack_s3e          = 0.99351


In [5]:
names = list(base_oof.keys())
oofs  = [base_oof[n] for n in names]
tes   = [base_te[n]  for n in names]

def neg_f1(w, oofs, y):
    w = np.maximum(w, 0)
    if w.sum() < 1e-9: return 0.0
    w = w / w.sum()
    p = sum(wi*oi for wi, oi in zip(w, oofs))
    return -f1_score(y, p.argmax(1), average='macro')

best_w, best_s = None, 0.0
for trial in range(50):
    np.random.seed(trial)
    w0 = np.random.dirichlet(np.ones(len(names)))
    res = minimize(neg_f1, w0, args=(oofs, y_true),
                   method='Nelder-Mead',
                   options={'xatol':1e-5,'fatol':1e-6,'maxiter':8000})
    if -res.fun > best_s:
        best_s = -res.fun
        best_w = np.maximum(res.x, 0); best_w /= best_w.sum()

print(f"Optimized blend OOF F1: {best_s:.5f}")
print(f"Weights:")
for n, w in zip(names, best_w):
    print(f"  {n:18s} = {w:.3f}")

blend_oof = sum(w*o for w,o in zip(best_w, oofs))
blend_te  = sum(w*t for w,t in zip(best_w, tes))

Optimized blend OOF F1: 0.99495
Weights:
  bb_base            = 0.069
  bb_multi           = 0.004
  muril_L            = 0.180
  eva02_L            = 0.443
  mm_eva_muril       = 0.233
  pl_muril           = 0.071


In [6]:
best_a, best_f = 0.0, 0.0
for a in np.linspace(0, 1, 201):
    f = f1_score(y_true, (a*blend_oof + (1-a)*stack_oof).argmax(1), average='macro')
    if f > best_f: best_f, best_a = f, a

mix_oof = best_a*blend_oof + (1-best_a)*stack_oof
mix_te  = best_a*blend_te  + (1-best_a)*stack_te
print(f"Best mix: blend_w={best_a:.3f}  stacker_w={1-best_a:.3f}  →  OOF F1 {best_f:.5f}")

Best mix: blend_w=0.855  stacker_w=0.145  →  OOF F1 0.99495


In [7]:

def opt_bias(probs, y, grid=np.arange(-0.5, 0.501, 0.01), passes=8):
    logp = np.log(probs + EPS)
    bias = np.zeros(8)
    def s(b): return f1_score(y, (logp+b).argmax(1), average='macro')
    for _ in range(passes):
        for c in range(8):
            orig = bias[c]; best_loc, best_loc_s = orig, s(bias)
            for d in grid:
                bias[c] = orig + d
                v = s(bias)
                if v > best_loc_s:
                    best_loc_s, best_loc = v, bias[c]
            bias[c] = best_loc
    return bias, s(bias)

# Single-shot (aggressive)
b_single, f_single = opt_bias(mix_oof, y_true)
print(f"Single-shot bias OOF F1: {f_single:.5f}")
print(f"Bias: {dict(zip(LABELS, np.round(b_single,3).tolist()))}")

# Bootstrap median (stable)
print("\nRunning bootstrap (15 iters)...")
n = len(y_true)
boot = []
for seed in range(15):
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, n, replace=True)
    b, _ = opt_bias(mix_oof[idx], y_true[idx], passes=4)
    boot.append(b)
b_stable = np.median(np.stack(boot), axis=0)
f_stable = f1_score(y_true, (np.log(mix_oof+EPS)+b_stable).argmax(1), average='macro')
print(f"\nBootstrap-stable bias OOF F1: {f_stable:.5f}")
print(f"Bias: {dict(zip(LABELS, np.round(b_stable,3).tolist()))}")

Single-shot bias OOF F1: 0.99590
Bias: {'Drought': -0.44, 'Earthquake': -0.22, 'Flood': -0.37, 'Human Damage': 0.0, 'Landslides': 0.0, 'Non Disaster': 0.0, 'Tropical Storm': -0.35, 'Wildfire': -0.58}

Running bootstrap (15 iters)...

Bootstrap-stable bias OOF F1: 0.99557
Bias: {'Drought': -0.34, 'Earthquake': -0.22, 'Flood': -0.37, 'Human Damage': 0.0, 'Landslides': 0.0, 'Non Disaster': 0.0, 'Tropical Storm': -0.35, 'Wildfire': 0.0}


In [8]:
for tag, bias in [('A1_stable', b_stable), ('A2_aggressive', b_single)]:
    pred = (np.log(mix_te + EPS) + bias).argmax(1)
    sub = pd.DataFrame({
        'image_id': test['image_id'].values,
        SUB_LABEL_COL: [LABELS[i] for i in pred],
    })
    # Verify column order matches sample_submission EXACTLY
    assert list(sub.columns) == list(sample_sub.columns), \
        f"Column mismatch: {list(sub.columns)} vs {list(sample_sub.columns)}"
    path = WORK / f'submission_{tag}.csv'
    sub.to_csv(path, index=False)
    print(f"\n=== {tag} ===")
    print(f"Saved: {path}")
    print(sub[SUB_LABEL_COL].value_counts().to_dict())

# Canonical submission Kaggle expects (rename the safer one)
import shutil
shutil.copy(WORK/'submission_A1_stable.csv', WORK/'submission.csv')
print("\n→ submission.csv (default Kaggle pickup) = A1_stable")


=== A1_stable ===
Saved: /kaggle/working/submission_A1_stable.csv
{'Earthquake': 202, 'Non Disaster': 201, 'Flood': 201, 'Drought': 200, 'Tropical Storm': 199, 'Landslides': 199, 'Human Damage': 197, 'Wildfire': 181}

=== A2_aggressive ===
Saved: /kaggle/working/submission_A2_aggressive.csv
{'Earthquake': 202, 'Non Disaster': 201, 'Flood': 201, 'Tropical Storm': 200, 'Human Damage': 200, 'Landslides': 199, 'Drought': 199, 'Wildfire': 178}

→ submission.csv (default Kaggle pickup) = A1_stable


In [9]:
final_pred = (np.log(mix_oof + EPS) + b_stable).argmax(1)
print("=== OOF report (A1_stable bias) ===")
print(classification_report(y_true, final_pred, target_names=LABELS, digits=4))

# Save for Phase 4 reuse
np.save(WORK/'phase1_mix_oof.npy',  mix_oof)
np.save(WORK/'phase1_mix_test.npy', mix_te)
np.save(WORK/'phase1_blend_w.npy',  best_w)
np.save(WORK/'phase1_b_stable.npy', b_stable)
np.save(WORK/'phase1_b_single.npy', b_single)
print("\nArtifacts saved for Phase 4.")

=== OOF report (A1_stable bias) ===
                precision    recall  f1-score   support

       Drought     0.9987    0.9950    0.9969       800
    Earthquake     1.0000    0.9888    0.9943       800
         Flood     0.9962    0.9925    0.9944       800
  Human Damage     0.9962    0.9962    0.9962       800
    Landslides     0.9853    0.9988    0.9920       803
  Non Disaster     0.9987    0.9975    0.9981       800
Tropical Storm     0.9963    0.9988    0.9975       800
      Wildfire     0.9931    0.9972    0.9951       720

      accuracy                         0.9956      6323
     macro avg     0.9956    0.9956    0.9956      6323
  weighted avg     0.9956    0.9956    0.9956      6323


Artifacts saved for Phase 4.


In [ ]:
import re
import numpy as np
import pandas as pd
 
CLASSES = ['Drought','Earthquake','Flood','Human Damage',
           'Landslides','Non Disaster','Tropical Storm','Wildfire']
L2I = {c: i for i, c in enumerate(CLASSES)}
 

TEXT_PROBS = [bb_base_test, bb_multi_test, muril_L_test, pl_muril_test] 
SUB_COL    = submission.columns[1]         
 
text_ens = sum(TEXT_PROBS) / len(TEXT_PROBS)
 
# --- the rule --------------------------------------------------------
AFTER_KW = re.compile(r'(দাবানলের|খরার|বন্যার|ঝড়ের|ভূমিকম্পের)\s*পর')  
KEYWORD_MAP = {'দাবানল': 'Wildfire',   
               'খরা':    'Drought'}    
CONF_TH = 0.90
labels = submission[SUB_COL].values.astype(object)
n_changed = 0
for i, ctx in enumerate(test['context'].astype(str).values):
    if AFTER_KW.search(ctx):           
        continue
    for kw, lab in KEYWORD_MAP.items():
        cid = L2I[lab]
        if kw in ctx and text_ens[i].argmax() == cid and text_ens[i, cid] > CONF_TH:  # guard (2)
            if labels[i] != lab:
                n_changed += 1
            labels[i] = lab
 
submission[SUB_COL] = labels
print(f"Text-feature rule overrode {n_changed} predictions.")
submission.to_csv('submission_final.csv', index=False)
print(submission[SUB_COL].value_counts().to_dict())